# Grounding DINO Fine-Tuning Setup
This notebook will download the `IDEA-Research/grounding-dino-base` model and save it to the local `./local_gdino_model` directory. After running this, your `main.py` will automatically load the model from this directory instead of HuggingFace.

It also contains boilerplate code demonstrating how to set up a fine-tuning loop for this model using PyTorch and your custom dataset.

In [ ]:
import os
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

model_id = "IDEA-Research/grounding-dino-base"
local_dir = "./local_gdino_model"

print(f"Downloading model {model_id} from HuggingFace...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id)

print(f"\nSaving model locally to {local_dir}...")
os.makedirs(local_dir, exist_ok=True)
processor.save_pretrained(local_dir)
model.save_pretrained(local_dir)
print("\nModel saved successfully! You can now restart your FastAPI backend to use this local directory.")

## Fine-Tuning Boilerplate
If you want to fine-tune Grounding DINO on your specific products (e.g. Fanta, Coca-Cola, Sprite), you can use the following PyTorch loop as a starting point. Grounding DINO fine-tuning is complex because it uses both vision and text encoders, but the most common approach is to freeze the backbones and only fine-tune the bounding box and class embedding heads.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

class CustomPlanogramDataset(Dataset):
    def __init__(self, processor, image_dir, annotation_file):
        self.processor = processor
        self.image_dir = image_dir
        
        # TODO: Load your dataset annotations here (e.g., from a JSON file)
        # Format should be a list of dictionaries like:
        # {
        #   "image_path": "img1.jpg", 
        #   "bboxes": [[10, 10, 100, 100]], # [x1, y1, x2, y2] format
        #   "labels": ["fanta"] # The ground truth text label
        # }
        self.data = []

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # 1. Load Image using PIL or OpenCV
        # from PIL import Image
        # image = Image.open(os.path.join(self.image_dir, item["image_path"])).convert("RGB")
        image = None # Placeholder
        
        # 2. Prepare text prompt (e.g. "fanta . coca-cola .")
        text_prompt = " . ".join(list(set(item["labels"]))) + " ."
        
        # 3. Process inputs
        # inputs = self.processor(images=image, text=text_prompt, return_tensors="pt")
        
        # 4. Return processed inputs along with target bounding boxes
        # return inputs, item["bboxes"]
        pass

def setup_fine_tuning():
    print("Loading local model for fine-tuning...")
    local_dir = "./local_gdino_model"
    processor = AutoProcessor.from_pretrained(local_dir)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(local_dir)

    # ---------------------------------------------------------
    # FREEZE BACKBONES (Vision & Text)
    # ---------------------------------------------------------
    for param in model.parameters():
        param.requires_grad = False  # Freeze everything first

    # UNFREEZE HEADS (Bounding Box and Class Embedding)
    for param in model.bbox_embed.parameters():
        param.requires_grad = True
    for param in model.class_embed.parameters():
        param.requires_grad = True

    print("Trainable parameters set up. Only the bounding box and classification heads will be updated.")

    # Setup Optimizer
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
    
    return model, processor, optimizer

# Uncomment below to initialize the setup:
# model, processor, optimizer = setup_fine_tuning()
